In [22]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 10000

In [23]:
df = pd.DataFrame({
    "customer_id": range(1, n + 1),

    "age": np.random.randint(
        21, 61, n
    ),

    "monthly_income": np.random.randint(
        15000, 150000, n
    ),

    "employment_type": np.random.choice(
        ["salaried", "self_employed", "contract"],
        n,
        p=[0.65, 0.25, 0.10]
    ),

    "credit_score": np.random.randint(
        500, 851, n
    ),

    "existing_emi": np.random.randint(
        0, 50000, n
    ),

    "loan_amount": np.random.randint(
        20000, 500000, n
    ),

    "loan_tenure": np.random.choice(
        [6, 12, 18, 24, 36],
        n
    ),

    "previous_default": np.random.choice(
        [0, 1],
        n,
        p=[0.85, 0.15]
    )
})

In [24]:
df.head()

,customer_id,age,monthly_income,employment_type,credit_score,existing_emi,loan_amount,loan_tenure,previous_default
0,1,59,118064,salaried,588,41834,306807,36,1
1,2,49,141417,self_employed,622,32981,337399,18,0
2,3,35,131585,salaried,546,29789,163458,24,0
3,4,28,45346,salaried,645,14905,434703,6,0
4,5,41,138152,salaried,502,30938,282360,24,0


In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customer_id       10000 non-null  int64 
 1   age               10000 non-null  int32 
 2   monthly_income    10000 non-null  int32 
 3   employment_type   10000 non-null  object
 4   credit_score      10000 non-null  int32 
 5   existing_emi      10000 non-null  int32 
 6   loan_amount       10000 non-null  int32 
 7   loan_tenure       10000 non-null  int64 
 8   previous_default  10000 non-null  int64 
dtypes: int32(5), int64(3), object(1)
memory usage: 507.9+ KB


In [26]:
df["loan_to_income"] = (
    df["loan_amount"] /
    (df["monthly_income"] * 12)
)

In [27]:
df["emi_to_income"] = (
    df["existing_emi"] /
    df["monthly_income"]
)

In [28]:
risk_score = (
    -0.004 * (df["credit_score"] - 650)
    + 2.5 * df["loan_to_income"]
    + 3.0 * df["emi_to_income"]
    + 1.5 * df["previous_default"]
    - 0.00001 * df["monthly_income"]
)

In [29]:
probability = 1 / (1 + np.exp(-risk_score))

In [30]:
df["default"] = np.random.binomial(
    1,
    probability
)

In [31]:
df["default"].value_counts(normalize=True)

default
1    0.6862
0    0.3138
Name: proportion, dtype: float64

In [32]:
missing_cols = [
    "monthly_income",
    "credit_score",
    "existing_emi"
]

for col in missing_cols:
    mask = np.random.rand(n) < 0.05
    df.loc[mask, col] = np.nan

In [33]:
df.isnull().sum()

customer_id           0
age                   0
monthly_income      482
employment_type       0
credit_score        527
existing_emi        488
loan_amount           0
loan_tenure           0
previous_default      0
loan_to_income        0
emi_to_income         0
default               0
dtype: int64

In [34]:
df.to_csv(
    "../data/customer_underwriting.csv",
    index=False
)

In [35]:
df.shape

(10000, 12)

In [36]:
df.describe()

,customer_id,age,monthly_income,credit_score,existing_emi,loan_amount,loan_tenure,previous_default,loan_to_income,emi_to_income,default
count,10000.00000,10000.000000,9518.000000,9473.000000,9512.000000,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,5000.50000,40.561800,82049.780836,673.672226,25194.531539,260434.27950,19.079400,0.149100,0.374796,0.433181,0.686200
std,2886.89568,11.454986,39326.149695,101.039157,14429.016761,139079.74988,10.283009,0.356205,0.366572,0.439723,0.464059
min,1.00000,21.000000,15005.000000,500.000000,0.000000,20037.00000,6.000000,0.000000,0.012329,0.000000,0.000000
25%,2500.75000,31.000000,47179.000000,585.000000,12758.000000,142002.25000,12.000000,0.000000,0.142161,0.153277,0.000000
50%,5000.50000,41.000000,82099.500000,675.000000,25369.500000,260405.50000,18.000000,0.000000,0.265264,0.306034,1.000000
75%,7500.25000,50.000000,116125.500000,760.000000,37698.250000,381648.50000,24.000000,0.000000,0.460261,0.532458,1.000000
max,10000.00000,60.000000,149997.000000,850.000000,49990.000000,499985.00000,36.000000,1.000000,2.735562,3.264741,1.000000


In [37]:
df["default"].value_counts()

default
1    6862
0    3138
Name: count, dtype: int64

In [38]:
df.groupby("employment_type")["default"].mean()

employment_type
contract         0.664115
salaried         0.685719
self_employed    0.696527
Name: default, dtype: float64

In [39]:
df.groupby(
    pd.cut(df["credit_score"], bins=[500, 600, 700, 800, 900])
)["default"].mean()

C:\Users\RISHABH123\AppData\Local\Temp\ipykernel_15312\274501926.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(


credit_score
(500, 600]    0.760901
(600, 700]    0.702642
(700, 800]    0.637486
(800, 900]    0.589327
Name: default, dtype: float64

In [40]:
X = df.drop(
    columns=["default", "customer_id"]
)

y = df["default"]

In [41]:
numeric_features = [
    "age",
    "monthly_income",
    "credit_score",
    "existing_emi",
    "loan_amount",
    "loan_tenure",
    "previous_default",
    "loan_to_income",
    "emi_to_income"
]

categorical_features = [
    "employment_type"
]

In [44]:
pip install scikit-learn


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\RISHABH123\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [45]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [46]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [47]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [48]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

In [49]:
full_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", model)
])

In [50]:
predictions = full_pipeline.predict(X_test)

NotFittedError: Pipeline is not fitted yet.